### 📚 Importing Libraries

🧰 Here, we're using import statements to bring in various libraries. os and json are standard Python libraries for operating system interactions and JSON handling

##### Data Visualization Tools 📈

For data visualization, matplotlib.pyplot and seaborn are imported. These libraries are great for creating a wide range of static, interactive, and animated plots 📊

##### 🐼 Data Handling with Pandas
pandas is a powerful data manipulation library in Python, and we're importing it as pd for ease of use in data processing and analysis tasks 

##### Setting Up a Dash Application 🖥️ 
dash is a productive Python framework for building web applications. We import dash itself, along with html and dcc from dash, which are used for creating HTML components and interactive elements 🌐

##### 📉 Interactive Data Visualization with Plotly
plotly.express is a plotting library that makes complex plots from dataframes. The px abbreviation is commonly used for brevity 

##### Building Interactive Dash Components 🔌 
The Input and Output from dash.dependencies are imported for creating interactive UI components that can update in response to user inputs 💡

In [1]:
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import dash
from dash import html, dcc
import plotly.express as px
from dash.dependencies import Input, Output

### Data processing 💯

#### Step 1️⃣

##### 📁 Setting the Data Directory
The data folder is where we expect to find the data files. 🗂️

##### 🔒 Safe Access to Nested Dictionary
safe_get is a function designed to safely access nested values in a dictionary. It attempts to traverse the dictionary using a list of keys and returns a default value if any key is missing or an error occurs. This is useful to avoid crashes due to missing data or structure changes in the dictionary. 🛡️

##### 🕒 Extracting Timestamp from File Name
The extract_timestamp function takes a file name, processes it to extract a timestamp, and then returns the timestamp. It's particularly useful for handling file names that encode date and time information, ensuring that data is accurately tagged with the time of its creation or modification. ⏱️

In [2]:
data_directory = '../data/'

def safe_get(dictionary, keys, default=None):
    for key in keys:
        try:
            dictionary = dictionary[key]
        except (TypeError, KeyError):
            return default
    return dictionary

def extract_timestamp(file_name):
    parts = file_name.rstrip('.jsonl').split('_')
    timestamp= '_'.join(parts[-2:])
    return timestamp

#### Step 2️⃣

##### 🔍 Defining the Data Processing Function
process_jsonl_files is a function that processes .jsonl files in a specified directory. This function is crucial for handling data related to different transportation modes like walking, driving, bicycling, and transit. 🚶‍♂️🚗🚴‍♀️🚌

##### 📂 Organizing Data into Categories
The function starts by creating a dictionary called categories with keys for different modes of transportation. Each key maps to an empty list intended to hold data specific to that category. 🗂️

##### 📚 File Iteration and Category Matching
It iterates through all files in the given directory. If a file ends with .jsonl, the function extracts its timestamp and checks if the file name includes any of the transportation categories. 🔄

##### 🔎 Reading and Extracting Data from Files
For each relevant file, it opens and reads the file line by line. Each line is a JSON string, which is loaded into a dictionary. It then extracts information about routes and legs of each route. 📖

##### 📌 Extracting Specific Data Points
For each leg of a route, it extracts the duration and start address using the safe_get function to handle potential data inconsistencies. The duration is converted to minutes and then stored in the corresponding category list along with the timestamp and start address. ⏱️

##### 🚫 Error Handling
The function includes error handling to catch and print any exceptions that occur while processing a file. This ensures that one problematic file won't stop the entire process and allows for debugging individual files. 🛠️

##### 🔢 Returning Processed Data
Once all files are processed, the function returns the categories dictionary. This dictionary contains categorized data, which can be used for further analysis or visualization. 📊

##### 🔧 Executing the Function
Finally, categorized_durations = process_jsonl_files(data_directory) calls the function with the specified data directory. This line starts the process of reading and categorizing data from .jsonl files stored in the given directory. 🏁

In [3]:
def process_jsonl_files(directory):
    categories = {
        'walking': [],
        'driving': [],
        'bicycling': [],
        'transit_bus': [],
        'transit_subway': []
    }
    
    for file_name in os.listdir(directory):
        if file_name.endswith('.jsonl'):
            timestamp = extract_timestamp(file_name) 
            for category in categories.keys():
                if category in file_name:
                    with open(os.path.join(directory, file_name), 'r') as file:
                        for line in file:
                            try:
                                data = json.loads(line)
                                routes = data.get('routes', [])
                                for route in routes:
                                    legs = route.get('legs', [])
                                    for leg in legs:
                                        duration = safe_get(leg, ['duration', 'value'])
                                        start_address = safe_get(leg, ['start_address'])
                                        if duration is not None:
                                            categories[category].append({
                                                'Timestamp': timestamp,
                                                'Residence': start_address,
                                                'Duration': duration / 60  
                                            })
                            except Exception as e:
                                print(f"Error in file {file_name}: {e}")
                    break
    return categories

categorized_durations = process_jsonl_files(data_directory)

#### Step 3️⃣

##### 🗺️ Defining Modes of Transportation

The variable modes is a list containing different modes of transportation like walking, bicycling, transit_bus, driving, and transit_subway. This categorization is essential for analyzing and comparing data across different travel methods 🚶‍♂️ 🚴‍♀️ 🚌 🚗 🚇

##### 🏠 Listing Residences
Residences is a list of strings, each representing a physical address in London. These addresses are specific locations likely relevant to the data analysis, possibly representing common starting or ending points for travel routes. 🏢

##### 📍 Naming the Residences
Correspondingly, residences_names provides human-readable names for each of these addresses. This makes the data more understandable and relatable, especially when presenting it to users who may be familiar with these names but not their associated addresses. 🏷️

##### 🔍 Ensuring Data Consistency
The assert statement checks that the lengths of residences and residences_names are equal. This is a crucial step to ensure data integrity, as it verifies that each address has a corresponding name. ✅

##### 📋 Creating a Mapping Dictionary
Finally, `address_to_name is a dictionary comprehension that maps each address to its corresponding name. This creates a useful reference that can quickly convert a long, detailed address into a more recognizable and concise residence name. This mapping is particularly helpful for data presentation and analysis, making the information more accessible and user-friendly. 🗝️

In [5]:
modes = ['walking', 'bicycling', 'transit_bus', 'driving', 'transit_subway']

residences = ["203 Westminster Bridge Rd, London SE1 7FR, UK",
             "Lilian Knowles House, 50 Crispin St, London E1 6HQ, UK",
             "Malet St, London WC1E 7HZ, UK",
             "Lansdowne Terrace, London WC1N 1AS, UK",
             "11 Gainsford St, London SE1 2NE, UK",
             "Bankside House, London SE1 9JA, UK",
             "18-24 Fitzroy St, London W1T 4BN, UK",
             "159 Great Dover St, London SE1 4WW, UK",
             "Connaught Hall, London WC1H 9EX, UK", 
             "178 High Holborn, London WC1V 7AA, UK",
             "Passfield Hall, London WC1H 0PW, UK",
             "University of, Nutford House, Brown St, London W1H 5UL, UK",
             "1 Cartwright Gardens, London WC1H 9EN, UK",
             "90 Rosebery Ave, London EC1R 4TY, UK"]  

residences_names = ["Urbanest Westminster Bridge",
    "Lilian Knowles House",
    "College Hall",
    "International Hall",
    "LSE Butler's Wharf",
    "Bankside House",
    "LSE Carr-Saunders Hall",
    "Unite Students - Sidney Webb House",
    "Connaught Hall",
    "LSE High Holborn",
    "Passfield Hall",
    "Nutford House",
    "Garden Halls",
    "Rosebery Hall"]

assert len(residences) == len(residences_names)
address_to_name = {address: name for address, name in zip(residences, residences_names)}

#### Step 4️⃣

##### 🔄 Iterating Over Modes and Residences

This loop iterates over each transportation mode and residence address. By doing this, it systematically processes data specific to each mode of transport for each residence location 🛤️ 

##### 📛 Assigning Residence Names
Inside the loop, each residence address is converted to its more recognizable name using the address_to_name mapping. This enhances the readability of the data by replacing the lengthy addresses with their corresponding residence names 🏠

##### 🔑 Creating Unique Keys for Dataframes
A unique key is generated for each combination of residence name and transportation mode, formatted as "ResidenceName_Mode". This key serves as an identifier for the specific subset of data in the final dataframes dictionary. 🔏

##### 🔍 Filtering Data for Each Combination
The code filters the categorized_durations data for entries that match both the current mode of transport and the specific residence. This ensures that each dataframe contains data relevant only to a particular combination of residence and mode

##### 📈 Creating and Storing Dataframes
If there is filtered data, the residence address in each data entry is replaced with the residence name for consistency and readability. Then, a pandas DataFrame is created from this filtered data and stored in the all_dataframes dictionary with the unique key 🏢➡️🏷️

In [6]:
all_dataframes = {}

for mode in modes:
    for residence in residences:
        residence_name = address_to_name[residence]
        key = f"{residence_name}_{mode}"
        filtered_data = [data for data in categorized_durations[mode] if data['Residence'] == residence]

        if filtered_data:
            for data in filtered_data:
                data['Residence'] = residence_name
            all_dataframes[key] = pd.DataFrame(filtered_data)

### 🧸 Graphics 

#### 🪡 Linegraph 

##### 🌐 Initializing the Dash Application

app = dash.Dash(__name__) initializes a new Dash web application. Dash is a Python framework for building interactive web applications. 🚀

##### 🖼️ Creating the App Layout
app.layout is set to a html.Div containing two dcc.Dropdown components and a dcc.Graph component. The dropdowns allow users to select a transportation mode and a residence, while the graph displays the relevant data based on these selections.

##### 🔽 Dropdown for Mode Selection
The first dropdown (mode-selector) is populated with the different modes of transportation (like walking, bicycling, etc.). This lets users choose which mode's data they want to see 🚶‍♂️ 🚴‍♀️ 🚌

##### 🔽 Dropdown for Residence Selection
The second dropdown (residence-selector) contains options for different residences. This enables users to filter the data based on the selected residence. 🏠

##### 📈 Setting Up the Graph
dcc.Graph(id='time-series-chart') is a placeholder for the interactive chart that will display the data. This graph will update based on the user's selections in the dropdowns. 📉

##### 🔄 Callback Function for Data Updates
The @app.callback decorator defines a function update_chart that updates the graph based on the selected mode and residence. This function gets triggered every time a user makes a selection. 🔄

##### 📊 Generating the Time Series Chart
Inside update_chart, the function constructs a key to access the relevant DataFrame from all_dataframes based on the selected mode and residence. It then checks if the data is valid and if essential columns ('Timestamp' and 'Duration') are present. 🗝️

##### 🔧 Preparing and Displaying the Chart
If valid data is available, px.line is used to create a line chart, with 'Timestamp' on the x-axis and 'Duration' on the y-axis. The chart title dynamically reflects the chosen mode and residence. 📈

##### 🎨 Customizing the Chart Layout
The layout of the chart is updated for better readability, particularly adjusting the x-axis settings to enhance the display of timestamps. 🖌️

##### 🌐 Running the Dash Server
Finally, if __name__ == '__main__': app.run_server(debug=True) starts the Dash server. When this script is run, it will launch a local web server and serve the Dash app, which can be accessed through a web browser. 

##### 📈 About the Visualized Chart

The line chart displays durations of various routes over time for a selected transportation mode and residence. The x-axis represents different timestamps (likely dates or times), and the y-axis shows the duration (presumably in minutes) it takes to complete a route using the selected mode from the chosen residence. This visualization helps to easily compare and analyze the efficiency or popularity of different modes of transportation from various residences over time.

In [7]:
app = dash.Dash(__name__)

app.layout = html.Div([
    dcc.Dropdown(
        id='mode-selector',
        options=[{'label': mode.capitalize(), 'value': mode} for mode in modes],
        value=modes[0],
        clearable=False,
        style={'width': '48%', 'display': 'inline-block'}
    ),
    dcc.Dropdown(
        id='residence-selector',
        options=[{'label': residence_name, 'value': residence_name} for residence_name in residences_names],
        value=residences[0],
        clearable=False,
        style={'width': '48%', 'display': 'inline-block', 'margin-left': '4%'}
    ),
    dcc.Graph(id='time-series-chart')
])

@app.callback(
    Output('time-series-chart', 'figure'),
    [Input('mode-selector', 'value'),
     Input('residence-selector', 'value')]
)

def update_chart(selected_mode,selected_residence):
    
    key = f"{selected_residence}_{selected_mode}"

    data = all_dataframes.get(key, pd.DataFrame())

    for key, df in all_dataframes.items():
        all_dataframes[key] = df.sort_values(by='Timestamp')

    if data.empty or 'Timestamp' not in data.columns or 'Duration' not in data.columns:
        return px.line(title='No data available')

    fig = px.line(
            data, 
            x='Timestamp',
            y='Duration',
            title=f"{selected_mode.capitalize()} Durations from {selected_residence}"
            )

    fig.update_layout(
         xaxis=dict(
         showticklabels=False,
         title_standoff=70,
         nticks=3500))       

    return fig

if __name__ == '__main__':
    app.run_server(debug=True, port = 8051)

#### Boxplot 1️⃣

##### 🌐 Initializing Dash App with Simplified Layout

In this revised version of the Dash app, the layout has been simplified to include only one dropdown (mode-selector) and a single dcc.Graph component (boxplot-chart). The dropdown now spans the entire width of the interface 🖼️

##### 🔽 Dropdown for Mode Selection
The dropdown allows users to select a mode of transportation. This selection determines the data to be displayed in the boxplot 🚴‍♂️

##### 🚗 Setting Up the Boxplot Graph
The dcc.Graph component, identified by boxplot-chart, is where the boxplot visualization will be displayed. This graph will dynamically update based on the selected transportation mode 📈

##### 🔄 Callback Function for Boxplot Updates
The @app.callback decorator links the dropdown selection to the boxplot generation. When a user selects a mode of transportation, the update_boxplot function is triggered 🔄

##### 🚌 Generating the Boxplot
update_boxplot function combines data from all residences for the selected transportation mode into one DataFrame. It then checks for the presence of necessary data (like 'Duration') before proceeding 📋

##### 📈 Creating and Customizing the Boxplot
If valid data is available, px.box from Plotly is used to create a boxplot. This boxplot visualizes the distribution of 'Duration' for different residences in the selected transportation mode. The plot includes all data points (points="all") and hover data showing timestamps 📦

##### 📉 About the Visualized Boxplot

The boxplot provides a statistical summary of travel durations for each residence in the selected mode of transportation.

For each residence, the boxplot displays the median, quartiles, and any outliers in travel duration.
The hover functionality allows users to see specific data points, including the exact duration and the timestamp for each trip.

This visualization is particularly useful for identifying variations in travel time, detecting outliers, and understanding the overall distribution of travel durations for each residence within a selected mode 🕒 📊 🏠

In [8]:
app = dash.Dash(__name__)

app.layout = html.Div([
    dcc.Dropdown(
        id='mode-selector',
        options=[{'label': mode.capitalize(), 'value': mode} for mode in modes],
        value=modes[0],
        clearable=False,
        style={'width': '100%', 'display': 'inline-block'}
    ),
    dcc.Graph(id='boxplot-chart')
])

@app.callback(
    Output('boxplot-chart', 'figure'),
    [Input('mode-selector', 'value')]
)
def update_boxplot(selected_mode):

    data_list = [df for key, df in all_dataframes.items() if key.endswith(selected_mode)]

    data = pd.concat(data_list, ignore_index=True)
    
    
    if data.empty or 'Duration' not in data.columns:
        return px.box(title='No data available')

    fig = px.box(
        data, 
        x='Residence',
        y='Duration',
        points="all",  
        title=f"Durations for {selected_mode.capitalize()} Mode",
        hover_data=['Timestamp']
    )

#     # Customizing hover template to show the timestamp more clearly
#     fig.update_traces(
#     hovertemplate="<br>".join([
#         "Residence: %{x}",
#         "Duration: %{y}",
#         "Timestamp: %{customdata[0]}"
#     ])
# )
    return fig

if __name__ == '__main__':
    app.run_server(debug=True,port = 8052)

#### Boxplot 2️⃣

##### ✨ Updated Dropdown for Residence Selection
In this version, the dropdown (residence-selector) allows users to choose from different residences. The selection of a residence now drives the data displayed in the boxplot.

##### 🎟️ Boxplot Graph Setup
The dcc.Graph component (boxplot-chart) remains as the placeholder for the boxplot visualization. The graph updates based on the residence selected from the dropdown 📈

##### 🎊 Callback Function for Updating Boxplot
The @app.callback decorator connects the residence dropdown selection to the boxplot update function, update_boxplot. Selecting a different residence triggers a new boxplot visualization.

##### 🏏 Generating Boxplot for Selected Residence
The update_boxplot function constructs a DataFrame for each mode of transportation specific to the selected residence. It then combines these into a single DataFrame for the boxplot visualization 📋

##### 📈 Boxplot Visualization
The boxplot, created using px.box, now displays the distribution of durations for different modes of transportation at the selected residence. Each mode is represented by a different color, enhancing the comparison across modes 🌈

In [9]:
app = dash.Dash(__name__)

app.layout = html.Div([
    dcc.Dropdown(
        id='residence-selector',
        options=[{'label': residence_name, 'value': residence_name} for residence_name in residences_names],
        value=residences_names[0],
        clearable=False,
        style={'width': '100%', 'display': 'inline-block'}
    ),
    dcc.Graph(id='boxplot-chart')
])

@app.callback(
    Output('boxplot-chart', 'figure'),
    [Input('residence-selector', 'value')]
)
def update_boxplot(selected_residence):

    mode_data = {mode: all_dataframes.get(f"{selected_residence}_{mode}", pd.DataFrame()) for mode in modes}

    combined_data = pd.concat(mode_data, names=['Mode', 'Index']).reset_index(level=0)
    
    fig = px.box(
        combined_data,
        x='Mode',
        y='Duration',
        color='Mode', 
        title=f"Duration Distributions for {selected_residence}",
        hover_data=['Timestamp'] 
    )
    return fig

if __name__ == '__main__':
    app.run_server(debug=True, port = 8053)

#### 🔥 Heatmap 1️⃣

##### 📢 Updated Visualization to Heatmap
In this iteration of the Dash app, the visualization changes from a boxplot to a heatmap. This offers a different perspective on the data, focusing on the average duration across different modes and residences 🌡️

##### 🎶 Heatmap Data Preparation
The update_heatmap function gathers and processes data for the selected residence across all transportation modes. It combines them into a single DataFrame, setting the stage for the heatmap visualization. 

##### 📽️ Creating the Heatmap
Using pd.concat, the function aggregates the durations data. Then, pivot_table is used to reshape this data into a format suitable for heatmap generation, with residences as rows, modes as columns, and average durations as values. 

##### 🖼️ Generating the Heatmap with Plotly Express
px.imshow creates the heatmap. This function is well-suited for displaying matrix-like data and is used here to visualize the average durations. The axes are labeled to clearly indicate modes of transport and residences, with the color intensity representing average duration 🎨

##### 🔧 Customizing Heatmap Appearance
The heatmap's appearance is fine-tuned for better readability and aesthetics. Adjustments include setting the x-axis to the bottom, turning off grid lines, and ensuring linear tick modes for both axes 🖌️

##### 🔖 Characteristics of the Heatmap Visualization

The heatmap provides a comparative view of average travel durations for each mode of transport from the selected residence.

Different colors indicate varying durations, with the color intensity (or heat) corresponding to longer or shorter average times.

This visualization is particularly useful for quickly identifying which modes of transport are typically faster or slower from a given residence, allowing for easy comparison across different modes.

In [10]:
app = dash.Dash(__name__)

app.layout = html.Div([
    dcc.Dropdown(
        id='residence-selector',
        options=[{'label': residence_name, 'value': residence_name} for residence_name in residences_names],
        value=residences_names[0],
        clearable=False,
        style={'width': '100%', 'display': 'inline-block'}
    ),
    dcc.Graph(id='boxplot-chart')
])

@app.callback(
    Output('boxplot-chart', 'figure'),
    [Input('residence-selector', 'value')]
)

def update_heatmap(selected_residence):
    durations_data = []
    for mode in modes:
        key = f"{selected_residence}_{mode}"
        if key in all_dataframes:
            df = all_dataframes[key]
            df['Mode'] = mode
            durations_data.append(df)
    
    combined_data = pd.concat(durations_data)
    
    heatmap_data = combined_data.pivot_table(
        index='Residence', 
        columns='Mode', 
        values='Duration', 
        aggfunc='mean'  # can change this to 'median' or other aggregation functions as needed
    )

    fig = px.imshow(
        heatmap_data,
        labels=dict(x="Mode", y="Residence", color="Average Duration"),
        x=heatmap_data.columns,
        y=heatmap_data.index, 
        title="Average Durations Heatmap"
    )
    
    fig.update_xaxes(side="bottom")
    fig.update_layout(
        xaxis_showgrid=False, yaxis_showgrid=False,
        xaxis=dict(tickmode='linear'), yaxis=dict(tickmode='linear')
    )

    return fig

if __name__ == '__main__':
    app.run_server(debug=True, port = 8054)

#### CO2 Bar chart 

In [12]:
app = dash.Dash(__name__)

app.layout = html.Div([
    dcc.Dropdown(
        id='residence-selector',
        options=[{'label': residence_name, 'value': residence_name} for residence_name in residences_names],
        value=residences_names[0],
        clearable=False,
        style={'width': '100%', 'display': 'inline-block'}
    ),
    dcc.Graph(id='bar-chart')
])

@app.callback(
    Output('bar-chart', 'figure'),
    [Input('residence-selector', 'value')]
)
def update_bar_chart(selected_residence):
    durations_data = []
    emissions_per_minute = {
        'bicycling': 0,
        'driving': 4.14, 
        'transit_bus': 2.85,  
        'transit_subway': 2.07,
        'walking': 0
    }
    
    for mode in modes:
        key = f"{selected_residence}_{mode}"
        if key in all_dataframes:
            df = all_dataframes[key]
            df['Mode'] = mode
            df['CO2_Emissions'] = df['Duration'] * emissions_per_minute.get(mode, 0)
            durations_data.append(df)
    
    combined_data = pd.concat(durations_data)
    
    bar_data = combined_data.groupby('Mode')['CO2_Emissions'].mean().reset_index()

    fig = px.bar(
        bar_data,
        x='Mode',
        y='CO2_Emissions',
        title="Average CO2 Emissions per Mode",
        labels={'CO2_Emissions': 'Average CO2 Emissions'}
    )

    fig.update_layout(xaxis_title="Transportation Mode", yaxis_title="Average CO2 Emissions")
    
    return fig

if __name__ == '__main__':
    app.run_server(debug=True, port=8056)
